# Big Data Challenge (BDC) Satria Data 2026
## Notebook Klasifikasi Limbah Padat Tingkat Lanjut (Simple & Clean Pipeline)

**Strategi:**
1. **Multi-Architecture Ensemble:** ConvNeXt Tiny + EfficientNet B2
2. **Full Backbone Freeze:** Hanya melatih classifier head, super cepat & stabil.
3. **Resolusi 288x288:** Detail tinggi untuk membedakan limbah.
4. **5-Fold Cross Validation & 10 Epoch:** Latihan stabil & konsisten.
5. **Weighted Loss:** Menangani class imbalance (Electronic sedikit).

In [ ]:
# ============================================================
# 1. KONFIGURASI & DETEKSI PATH DATASET (Kaggle & Colab)
# ============================================================
import os
import sys
from pathlib import Path

def detect_environment():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return 'kaggle'
    try:
        import google.colab
        return 'colab'
    except ImportError:
        pass
    return 'local'

ENV = detect_environment()
print(f'Environment terdeteksi: {ENV}')

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = '/content/drive/MyDrive/BISMILLAH BDC 2026/BDC SatDat 2026'
    if os.path.exists(project_path):
        sys.path.append(project_path)
        os.chdir(project_path)
        print(f'Berhasil berpindah ke direktori project: {os.getcwd()}')

# Setup Path Dataset
if ENV == 'colab':
    if Path('/content/BDC2026/train').exists():
        DATA_DIR = Path('/content/BDC2026')
    elif Path('/content/drive/MyDrive/BISMILLAH BDC 2026/BDC2026/train').exists():
        DATA_DIR = Path('/content/drive/MyDrive/BISMILLAH BDC 2026/BDC2026')
    else:
        DATA_DIR = Path('/content/drive/MyDrive/BISMILLAH BDC 2026/BDC SatDat 2026/BDC2026')
elif ENV == 'kaggle':
    kaggle_input = Path('/kaggle/input')
    found = False
    for root, dirs, files in os.walk(kaggle_input):
        if 'train' in dirs:
            train_candidate = Path(root) / 'train'
            sub_items = os.listdir(train_candidate)
            if any('ecyclable' in s or 'lectronic' in s or 'rganic' in s for s in sub_items):
                DATA_DIR = Path(root)
                found = True
                break
    if not found:
        DATA_DIR = kaggle_input / 'bdc2026-dataset'
        print('WARNING: Folder train tidak ditemukan.')
else:
    DATA_DIR = Path('BDC2026')
    if not DATA_DIR.exists():
        DATA_DIR = Path('../BDC2026')

TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
SUBMISSION_TEMPLATE = DATA_DIR / 'submission.csv'

if ENV == 'kaggle':
    PROJECT_ROOT = Path('/kaggle/working')
elif ENV == 'colab':
    PROJECT_ROOT = Path('/content/drive/MyDrive/BISMILLAH BDC 2026/BDC SatDat 2026')
else:
    PROJECT_ROOT = Path('.')

OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
PLOTS_DIR = OUTPUTS_DIR / 'plots'
MODELS_DIR = PROJECT_ROOT / 'models'

for d in (OUTPUTS_DIR, PLOTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['Recyclable', 'Electronic', 'Organic']
CLASS_TO_IDX = {'Recyclable': 0, 'Electronic': 1, 'Organic': 2}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
SEED = 42

print(f'Data dir         : {DATA_DIR} (exists={DATA_DIR.exists()})')
print(f'Train dir        : {TRAIN_DIR} (exists={TRAIN_DIR.exists()})')
print(f'Test dir         : {TEST_DIR} (exists={TEST_DIR.exists()})')
print(f'Submission tmpl  : {SUBMISSION_TEMPLATE} (exists={SUBMISSION_TEMPLATE.exists()})')
print(f'Models dir       : {MODELS_DIR}')

In [ ]:
# 2. INSTALL LIBRARY
!pip install -q timm scikit-learn pandas pillow

In [ ]:
# 3. IMPORT LIBRARIES
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import glob
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# 4. PENGUMPULAN DATA LATIH
image_paths = []
labels = []

for subfolder in os.listdir(TRAIN_DIR):
    subfolder_path = TRAIN_DIR / subfolder
    if os.path.isdir(subfolder_path):
        class_idx = -1
        name_clean = subfolder.lower()
        if 'recyclable' in name_clean:
            class_idx = 0
        elif 'electronic' in name_clean:
            class_idx = 1
        elif 'organic' in name_clean:
            class_idx = 2
            
        if class_idx != -1:
            subfolder_images = []
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
                subfolder_images.extend(glob.glob(os.path.join(subfolder_path, ext)))
            
            unique_subfolder_images = sorted(list(set(os.path.abspath(p) for p in subfolder_images)))
            
            for img_path in unique_subfolder_images:
                filename = os.path.basename(img_path)
                if filename.startswith('.'):
                    continue
                image_paths.append(img_path)
                labels.append(class_idx)

df_train = pd.DataFrame({
    'image_path': image_paths,
    'label': labels
})

print(f'Total data latih: {len(df_train)} gambar')
print(df_train['label'].value_counts().rename(index=IDX_TO_CLASS))

In [ ]:
# 5. DATASET CLASS
class WasteDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = row['label']
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# 6. AUGMENTASI (Resolusi 288x288 untuk Detail Sirkuit Elektronik/Plastik)
IMG_SIZE = 288
BATCH_SIZE = 32

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(f'Image size: {IMG_SIZE} | Batch size: {BATCH_SIZE}')

In [ ]:
# 7. FUNGSI TRAINING ONE EPOCH & VALIDATE (Log Bersih Per Epoch)
def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler=None):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        if scaler is not None and device.type == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_f1

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    val_loss = running_loss / len(dataloader.dataset)
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    return val_loss, val_f1, all_preds, all_labels

In [ ]:
# 8. HELPER TRAINING K-FOLD (5-Fold, 10 Epoch)
def train_model_kfold(model_name, n_splits=5, num_epochs=10):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    fold_results = []
    
    log_file_path = PROJECT_ROOT / f'training_log_{model_name}.txt'
    with open(log_file_path, 'w') as f:
        f.write(f'=== LOG TRAINING {model_name} ===\n\n')
        
    def write_log(message):
        print(message)
        with open(log_file_path, 'a') as f:
            f.write(message + '\n')
            
    write_log(f'Memulai training {model_name} ({n_splits}-Fold)...\n')
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df_train, df_train['label'])):
        try:
            write_log('=' * 50)
            write_log(f' {model_name} | FOLD {fold + 1} / {n_splits}')
            write_log('=' * 50)
            
            fold_train_df = df_train.iloc[train_idx]
            fold_val_df = df_train.iloc[val_idx]
            
            fold_train_dataset = WasteDataset(fold_train_df, transform=train_transforms)
            fold_val_dataset = WasteDataset(fold_val_df, transform=val_transforms)
            
            # Multi-worker DataLoader untuk percepatan I/O di Kaggle/Colab
            NUM_WORKERS = 2 if ENV in ['kaggle', 'colab'] else 0
            train_loader = DataLoader(fold_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
            val_loader = DataLoader(fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
            
            model = timm.create_model(model_name, pretrained=True, num_classes=len(CLASS_NAMES))
            model = model.to(device)
            scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
            
            # Bobot loss untuk class imbalance (Recyclable: 9999, Electronic: 3961, Organic: 12567)
            class_weights = torch.tensor([1.25, 3.17, 1.00]).to(device)
            criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
            
            # --- Differential Learning Rate (Fine-Tuning Sejak Epoch 1) ---
            backbone_params = []
            head_params = []
            for name, param in model.named_parameters():
                param.requires_grad = True
                if any(k in name.lower() for k in ['head', 'fc', 'classifier']):
                    head_params.append(param)
                else:
                    backbone_params.append(param)
            
            optimizer = optim.AdamW([
                {'params': backbone_params, 'lr': 2e-5},
                {'params': head_params, 'lr': 1e-3}
            ], weight_decay=1e-2)
            
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
            
            best_val_f1 = 0.0
            best_model_path = MODELS_DIR / f'best_{model_name}_fold_{fold}.pth'
            
            for epoch in range(num_epochs):
                start_time = time.time()
                train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler=scaler)
                val_loss, val_f1, _, _ = validate(model, val_loader, criterion, device)
                
                scheduler.step()
                elapsed = time.time() - start_time
                
                msg = (f'Epoch {epoch+1}/{num_epochs} | '
                       f'Train Loss: {train_loss:.4f} - Train F1: {train_f1:.4f} | '
                       f'Val Loss: {val_loss:.4f} - Val F1: {val_f1:.4f} | '
                       f'Time: {elapsed:.2f}s')
                write_log(msg)
                
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    torch.save(model.state_dict(), best_model_path)
                    write_log(f'  ==> Saved best fold weights dengan Val F1: {best_val_f1:.4f}')
            
            fold_results.append(best_val_f1)
            write_log(f'\nFold {fold+1} selesai. Best Val Macro F1: {best_val_f1:.4f}\n')
        except Exception as e:
            write_log(f'Error pada fold {fold+1}: {str(e)}')
        
    write_log('=' * 50)
    write_log(f' TRAINING {model_name} K-FOLD SELESAI')
    write_log('=' * 50)
    write_log(f'Rata-rata Macro F1-Score: {np.mean(fold_results):.4f} (+/- {np.std(fold_results):.4f})\n')
    return fold_results

In [ ]:
# 9. PROSES TRAINING MULTI-MODEL (5-Fold, 10 Epoch)
N_SPLITS = 5
EPOCHS = 10

# Model A: ConvNeXt Tiny
convnext_results = train_model_kfold('convnext_tiny', n_splits=N_SPLITS, num_epochs=EPOCHS)

# Model B: EfficientNet B2
effnet_results = train_model_kfold('efficientnet_b2', n_splits=N_SPLITS, num_epochs=EPOCHS)

In [ ]:
# 10. ENSEMBLE INFERENCE (10 MODEL) & EXPORT SUBMISSION
model_names = ['convnext_tiny', 'efficientnet_b2']

print("Memuat 10 model terbaik untuk ensemble...")
models = []
for name in model_names:
    for fold in range(N_SPLITS):
        model_instance = timm.create_model(name, pretrained=False, num_classes=len(CLASS_NAMES))
        weights_path = MODELS_DIR / f'best_{name}_fold_{fold}.pth'
        if weights_path.exists():
            model_instance.load_state_dict(torch.load(weights_path, map_location=device))
            model_instance = model_instance.to(device)
            model_instance.eval()
            models.append(model_instance)
            print(f"-> Loaded best_{name}_fold_{fold}.pth")

print(f"Total model sukses termuat: {len(models)} model.")

# Koreksi ID String
df_submission = pd.read_csv(SUBMISSION_TEMPLATE)
df_submission['id'] = df_submission['id'].astype(float).astype(int).astype(str)

ensemble_predictions = []
softmax = nn.Softmax(dim=1)

total_test = len(df_submission)
with torch.no_grad():
    for idx, row in df_submission.iterrows():
        if (idx + 1) % 200 == 0 or (idx + 1) == total_test:
            print(f'Predicting: {idx + 1}/{total_test}', flush=True)
        test_id = row['id']
        img_name_candidates = [f'{test_id}.jpg', f'{test_id}.jpeg', f'{test_id}.png']
        img_path = None
        for cand in img_name_candidates:
            cand_path = TEST_DIR / cand
            if cand_path.exists():
                img_path = cand_path
                break
        
        if img_path is None:
            print(f'WARNING: Image dengan ID {test_id} tidak ditemukan.')
            ensemble_predictions.append(0)
            continue
            
        image = Image.open(img_path).convert('RGB')
        image = val_transforms(image).unsqueeze(0).to(device)
        
        # Soft voting
        accumulated_prob = torch.zeros(1, len(CLASS_NAMES)).to(device)
        for model_instance in models:
            outputs = model_instance(image)
            probs = softmax(outputs)
            accumulated_prob += probs
            
        final_pred = torch.argmax(accumulated_prob, dim=1).item()
        ensemble_predictions.append(final_pred)

df_submission['predicted'] = ensemble_predictions
output_file = PROJECT_ROOT / 'submission_SD2026040000187.csv'
df_submission.to_csv(output_file, index=False)

print(f'\nEnsemble Submission sukses disimpan di: {output_file}')
print(df_submission['predicted'].value_counts().rename(index=IDX_TO_CLASS))